# MMUbiPred–Context Residual Hybrid: manuscript visualizations

This notebook generates post hoc performance and biological-interpretation figures for the frozen residual hybrid. The performance section includes t-SNE, ROC curves, precision–recall curves, confusion-matrix comparison, and five-fold training/validation curves. The biological section examines local residue enrichment, expert complementarity, and broad-context composition. Saved probabilities are reused throughout. One frozen inference pass is performed only when hidden representations required by t-SNE have not already been cached. Every figure is displayed after generation and exported as PDF, SVG, and 600-DPI PNG. No training, threshold selection, model selection, or metric optimization occurs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Repository and model access

Private-repository access uses a fine-grained read-only GitHub token stored in Colab Secrets as `GITHUB_TOKEN`. An optional `HF_TOKEN` supports authenticated ESM-2 retrieval. The Hugging Face cache remains on Drive.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN, HF_TOKEN = secret('GITHUB_TOKEN'), secret('HF_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/content/drive/MyDrive/MMUbiPred-research/huggingface-cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import hashlib, json, numpy as np, torch
from IPython.display import display
import matplotlib.pyplot as plt

print('Device available:', 'cuda' if torch.cuda.is_available() else 'cpu')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

## Frozen-artifact preflight

The visualization uses the completed five-fold development run, frozen five-epoch Short-Range Expert refit, frozen seven-epoch Long-Context Expert refit, matched historical-test probabilities, and historical-test UniProt cache. The preflight reports every required path before feature extraction.

In [ ]:
CONFIG = Path('experiment/configs/mmubipred_context_residual_v1.json')
DEVELOPMENT_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-v1')
CONTEXT_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-only-final-v1-seed42')
HYBRID_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-full-refit-exploratory-v1')
TEST_SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/context/uniprot_test_sequences.json')
VISUALIZATION_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/visualizations/residual-hybrid-v1')
FIGURE_DIR = VISUALIZATION_DIR / 'figures'
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

required = {
    'development summary': DEVELOPMENT_RUN_DIR / 'stabilized_summary.json',
    'Short-Range checkpoint': HYBRID_RUN_DIR / 'best.pt',
    'hybrid metrics': HYBRID_RUN_DIR / 'residual_hybrid_test_metrics.json',
    'hybrid predictions': HYBRID_RUN_DIR / 'residual_hybrid_test_predictions.npz',
    'Long-Context Expert checkpoint': CONTEXT_RUN_DIR / 'best.pt',
    'Long-Context Expert metrics': CONTEXT_RUN_DIR / 'context_test_metrics.json',
    'matched comparison': CONTEXT_RUN_DIR / 'matched_mmubipred_comparison.json',
    'test sequence cache': TEST_SEQUENCE_CACHE,
}
for role, path in required.items():
    print(f'{role:24s}', 'FOUND' if path.exists() else 'MISSING', path)
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing frozen artifacts:\n' + '\n'.join(missing))

## Frozen hidden-representation extraction

The extraction is skipped when a completed cache exists. Otherwise, a GPU runtime is required for the LoRA-ESM-2 pass. The script verifies that recomputed probabilities reproduce the saved frozen probabilities before accepting either hidden matrix.

In [ ]:
REPRESENTATIONS_PATH = VISUALIZATION_DIR / 'matched_hidden_representations.npz'
REPRESENTATION_MANIFEST_PATH = VISUALIZATION_DIR / 'representation_manifest.json'

if REPRESENTATIONS_PATH.exists() and REPRESENTATION_MANIFEST_PATH.exists():
    print('Existing frozen representation cache found; inference was not repeated.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('A GPU runtime is required for the one-time ESM-2 representation extraction.')
    run_live([
        sys.executable, '-u',
        'experiment/scripts/extract_residual_hybrid_representations.py',
        '--config', CONFIG,
        '--hybrid-run-dir', HYBRID_RUN_DIR,
        '--context-run-dir', CONTEXT_RUN_DIR,
        '--test-sequence-cache', TEST_SEQUENCE_CACHE,
        '--output-dir', VISUALIZATION_DIR,
        '--batch-size', '32',
        '--num-workers', '0',
        '--allow-historical-test-visualization',
    ])

representation_manifest = json.loads(REPRESENTATION_MANIFEST_PATH.read_text())
if sha256_file(REPRESENTATIONS_PATH) != representation_manifest['output']['sha256']:
    raise RuntimeError('The frozen representation-cache checksum changed.')
display(representation_manifest)

## Load aligned visualization data

In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'experiment' / 'src'))
from ubipred.visualization import (
    compute_tsne,
    plot_confusion_matrix_comparison,
    plot_precision_recall_curves,
    plot_roc_curves,
    plot_training_validation_curves,
    plot_tsne_comparison,
    save_figure,
    stratified_visualization_indices,
)

with np.load(REPRESENTATIONS_PATH, allow_pickle=False) as saved:
    dataset_indices = saved['dataset_indices'].astype(np.int64)
    labels = saved['labels'].astype(np.int64)
    local_representations = saved['local_representations'].astype(np.float32)
    context_representations = saved['context_representations'].astype(np.float32)
    probabilities = {
        'Exact MMUbiPred': saved['paper_probabilities'].astype(np.float64),
        'Short-Range Expert': saved['local_probabilities'].astype(np.float64),
        'Long-Context Expert': saved['context_probabilities'].astype(np.float64),
        'Residual hybrid': saved['residual_probabilities'].astype(np.float64),
    }

assert local_representations.shape == (len(labels), 6)
assert context_representations.shape == (len(labels), 256)
assert all(values.shape == labels.shape for values in probabilities.values())
display({
    'cohort_size': len(labels),
    'negative_sites': int(np.sum(labels == 0)),
    'positive_sites': int(np.sum(labels == 1)),
    'Short-Range Expert representation shape': local_representations.shape,
    'Long-Context Expert representation shape': context_representations.shape,
})

## Figure 1 — t-SNE representation comparison

The same deterministic class-balanced subset is used in both panels. t-SNE coordinates are cached because the nonlinear embedding is computationally expensive. This figure is qualitative and is not used for model selection or performance claims.

In [ ]:
TSNE_PATH = VISUALIZATION_DIR / 'tsne_coordinates_seed42.npz'
TSNE_MAX_POINTS = 6000
TSNE_SEED = 42
TSNE_PERPLEXITY = 30.0
TSNE_ITERATIONS = 1500

sample_indices = stratified_visualization_indices(labels, TSNE_MAX_POINTS, TSNE_SEED)
if TSNE_PATH.exists():
    with np.load(TSNE_PATH, allow_pickle=False) as cached:
        cached_indices = cached['sample_indices'].astype(np.int64)
        if not np.array_equal(cached_indices, sample_indices):
            raise RuntimeError('Cached t-SNE sample indices do not match the frozen configuration.')
        local_tsne = cached['local_tsne']
        context_tsne = cached['context_tsne']
        if int(cached['seed']) != TSNE_SEED or float(cached['perplexity']) != TSNE_PERPLEXITY or int(cached['iterations']) != TSNE_ITERATIONS:
            raise RuntimeError('Cached t-SNE parameters do not match the frozen configuration.')
        if str(cached['representations_sha256']) != sha256_file(REPRESENTATIONS_PATH):
            raise RuntimeError('Cached t-SNE coordinates use a different representation artifact.')
    print('Existing t-SNE coordinates reused:', TSNE_PATH)
else:
    print('Computing Short-Range Expert t-SNE...', flush=True)
    local_tsne = compute_tsne(
        local_representations[sample_indices],
        seed=TSNE_SEED, perplexity=TSNE_PERPLEXITY, iterations=TSNE_ITERATIONS,
    )
    print('Computing Long-Context Expert t-SNE...', flush=True)
    context_tsne = compute_tsne(
        context_representations[sample_indices],
        seed=TSNE_SEED, perplexity=TSNE_PERPLEXITY, iterations=TSNE_ITERATIONS,
    )
    np.savez_compressed(
        TSNE_PATH,
        sample_indices=sample_indices,
        dataset_indices=dataset_indices[sample_indices],
        labels=labels[sample_indices],
        local_tsne=local_tsne,
        context_tsne=context_tsne,
        seed=np.asarray(TSNE_SEED),
        perplexity=np.asarray(TSNE_PERPLEXITY),
        iterations=np.asarray(TSNE_ITERATIONS),
        representations_sha256=np.asarray(sha256_file(REPRESENTATIONS_PATH)),
    )
display({
    'visualized_sites': len(sample_indices),
    'negative_sites': int(np.sum(labels[sample_indices] == 0)),
    'positive_sites': int(np.sum(labels[sample_indices] == 1)),
    'seed': TSNE_SEED,
    'perplexity': TSNE_PERPLEXITY,
    'iterations': TSNE_ITERATIONS,
})

In [ ]:
figure = plot_tsne_comparison(local_tsne, context_tsne, labels[sample_indices])
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_01_tsne_representations')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Figure 2 — ROC curves

All four models are compared on the identical 12,288-site matched cohort using their saved positive-class probabilities.

In [ ]:
figure = plot_roc_curves(labels, probabilities)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_02_roc_curves')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Figure 3 — Precision–recall curves

The horizontal reference line denotes positive-class prevalence in the matched cohort.

In [ ]:
figure = plot_precision_recall_curves(labels, probabilities)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_03_precision_recall_curves')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Figure 4 — Confusion-matrix comparison

Counts and row-normalized percentages are reported at the frozen threshold 0.5. Row normalization separates sensitivity from specificity despite class imbalance.

In [ ]:
figure = plot_confusion_matrix_comparison(labels, probabilities, threshold=0.5)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_04_confusion_matrices')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Figure 5 — Five-fold training and validation curves

The Short-Range Expert panels use the uniformly stabilized initialization selected within each fold. The Long-Context Expert panels use the original unchanged Long-Context Expert histories. Validation MCC is evaluated at threshold 0.5. These are development curves; the final five-epoch and seven-epoch full-data refits had no validation sets.

In [ ]:
local_histories, context_histories, selected_local_seeds = [], [], []
for fold in range(5):
    fold_dir = DEVELOPMENT_RUN_DIR / 'folds' / f'fold_{fold}'
    selection_path = fold_dir / 'local_stabilized' / 'selection.json'
    context_history_path = fold_dir / 'context' / 'development_history.json'
    if not selection_path.exists() or not context_history_path.exists():
        raise FileNotFoundError(f'Incomplete fold history: {fold_dir}')
    selection = json.loads(selection_path.read_text())
    selected_seed = int(selection['selected_initialization_seed'])
    local_history_path = (
        fold_dir / 'local_stabilized' / 'candidates' / f'seed_{selected_seed}' / 'development_history.json'
    )
    if not local_history_path.exists():
        raise FileNotFoundError(f'Missing selected Short-Range Expert history: {local_history_path}')
    selected_local_seeds.append(selected_seed)
    local_histories.append(json.loads(local_history_path.read_text()))
    context_histories.append(json.loads(context_history_path.read_text()))

histories = {
    'Short-Range Expert': local_histories,
    'Long-Context Expert': context_histories,
}
display({
    'folds': list(range(5)),
    'selected Short-Range Expert initialization seeds': selected_local_seeds,
    'Short-Range Expert epochs completed': [len(history) for history in local_histories],
    'Long-Context Expert epochs completed': [len(history) for history in context_histories],
})

In [ ]:
figure = plot_training_validation_curves(histories)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_05_training_validation_curves')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

## Output inventory

In [ ]:
generated = sorted(path for path in FIGURE_DIR.iterdir() if path.is_file())
expected_stems = {
    'figure_01_tsne_representations',
    'figure_02_roc_curves',
    'figure_03_precision_recall_curves',
    'figure_04_confusion_matrices',
    'figure_05_training_validation_curves',
}
expected_files = {f'{stem}.{suffix}' for stem in expected_stems for suffix in ('pdf', 'svg', 'png')}
observed_files = {path.name for path in generated}
if not expected_files.issubset(observed_files):
    raise RuntimeError(f'Missing figure exports: {sorted(expected_files - observed_files)}')
figure_manifest = {
    'status': 'five manuscript visualization figures generated',
    'analysis_role': 'post hoc visualization; no model or threshold selection',
    'cohort_records': len(labels),
    'matched_test_indices_sha256': representation_manifest['cohort']['matched_indices_sha256'],
    'representation_artifact_sha256': sha256_file(REPRESENTATIONS_PATH),
    'tsne': {
        'sample_size': len(sample_indices),
        'seed': TSNE_SEED,
        'perplexity': TSNE_PERPLEXITY,
        'iterations': TSNE_ITERATIONS,
        'coordinates_sha256': sha256_file(TSNE_PATH),
    },
    'classification_threshold': 0.5,
    'figures': {path.name: sha256_file(path) for path in generated},
}
FIGURE_MANIFEST_PATH = VISUALIZATION_DIR / 'figure_manifest.json'
FIGURE_MANIFEST_PATH.write_text(json.dumps(figure_manifest, indent=2, sort_keys=True) + '\n')
display({
    'figure_directory': str(FIGURE_DIR),
    'generated_files': [path.name for path in generated],
    'representation_cache': str(REPRESENTATIONS_PATH),
    'tsne_cache': str(TSNE_PATH),
    'figure_manifest': str(FIGURE_MANIFEST_PATH),
    'interpretation': figure_manifest['analysis_role'],
})

## Biological interpretation

This section asks which sequence characteristics are associated with the labels and with complementary expert predictions. It uses the same 12,288-site matched historical-test cohort, the released 49-residue windows, exact-match 257-residue UniProt contexts, and the frozen threshold of 0.5. These analyses are post hoc and descriptive: residue enrichment and expert complementarity can generate biological hypotheses, but they do not establish causal recognition motifs or molecular mechanisms. No result in this section is used to modify the model.

In [ ]:
from ubipred.context import build_validated_contexts, load_sequence_cache
from ubipred.fasta import load_released_split
from ubipred.visualization import (
    AMINO_ACIDS,
    plot_context_radial_enrichment,
    plot_expert_complementarity_biology,
    plot_local_sequence_enrichment,
    positional_log2_enrichment,
    radial_band_log2_enrichment,
)

experiment_config = json.loads(CONFIG.read_text())
data_dir = Path(experiment_config['data_dir'])
if not data_dir.is_absolute():
    data_dir = REPO_ROOT / data_dir
all_test_records, _ = load_released_split(data_dir, split='test', window_size=49)
matched_records = [all_test_records[int(index)] for index in dataset_indices]
local_sequences = [record.sequence for record in matched_records]

full_sequences, sequence_cache_metadata = load_sequence_cache(TEST_SEQUENCE_CACHE)
validated_indices, context_sequences, context_validation = build_validated_contexts(
    all_test_records,
    full_sequences,
    context_window_size=int(experiment_config['context_window_size']),
)
validated_indices = np.asarray(validated_indices, dtype=np.int64)
if not np.array_equal(validated_indices, dataset_indices):
    raise RuntimeError('Biological contexts do not reproduce the frozen matched cohort.')
if any(len(sequence) != 49 or sequence[24] != 'K' for sequence in local_sequences):
    raise RuntimeError('Invalid Short-Range lysine-centred windows were reconstructed.')
if any(len(sequence) != 257 or sequence[128] != 'K' for sequence in context_sequences):
    raise RuntimeError('Invalid Long-Context lysine-centred windows were reconstructed.')
if not np.array_equal(np.asarray([record.label for record in matched_records]), labels):
    raise RuntimeError('Biological inputs are not aligned with the saved labels.')

display({
    'analysis_role': 'post hoc descriptive biological interpretation',
    'matched_sites': len(labels),
    'unique_proteins': len({record.protein_id for record in matched_records}),
    'Short-Range Expert window': len(local_sequences[0]),
    'Long-Context Expert window': len(context_sequences[0]),
    'Long-Context validation': context_validation,
})

### Figure 6 — Local amino-acid enrichment around the central lysine

At every relative position, the heatmap reports the smoothed log2 ratio of residue frequency in labelled ubiquitinated sites to its frequency in labelled non-ubiquitinated sites. Red therefore indicates enrichment around positive sites and blue indicates depletion. The dashed line marks the invariant central lysine. A pseudocount of 0.5 is used, and displayed values are clipped at ±2 for legibility.

In [ ]:
figure = plot_local_sequence_enrichment(local_sequences, labels)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_06_local_residue_enrichment')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

### Figure 7 — Expert complementarity and associated local sequence signatures

Panel A partitions each true class by whether the Short-Range and Long-Context Experts were correct. Panels B and C compare residue frequencies among sites correctly classified by only one expert. Red denotes residues enriched among Long-Context-only successes; blue denotes residues enriched among Short-Range-only successes. Positive and negative sites are analyzed separately to prevent class prevalence from creating an apparent expert-specific signature.

In [ ]:
figure, complementarity_counts = plot_expert_complementarity_biology(
    local_sequences,
    labels,
    probabilities['Short-Range Expert'],
    probabilities['Long-Context Expert'],
    threshold=0.5,
)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_07_expert_complementarity_biology')
display(figure)
plt.close(figure)
display(complementarity_counts)
print('Saved:', *figure_paths, sep='\n')

### Figure 8 — Broad-context amino-acid composition

The 257-residue window is summarized in four symmetric distance bands around the target lysine. Each cell reports positive-versus-negative amino-acid enrichment after excluding padding and the central residue. The ±1–5 and ±6–24 bands describe the immediate and conventional local neighborhood; ±25–64 and ±65–128 describe sequence information unavailable to the paper's 49-residue input. This composition-level analysis does not attribute an individual prediction to a particular residue.

In [ ]:
CONTEXT_BANDS = ((1, 5), (6, 24), (25, 64), (65, 128))
figure = plot_context_radial_enrichment(context_sequences, labels, bands=CONTEXT_BANDS)
figure_paths = save_figure(figure, FIGURE_DIR, 'figure_08_context_radial_enrichment')
display(figure)
plt.close(figure)
print('Saved:', *figure_paths, sep='\n')

### Descriptive summary and output inventory

The table-like output below lists the largest observed enrichment values to support figure inspection. These rankings are exploratory and should be discussed as dataset-associated sequence patterns rather than universal ubiquitination motifs. Formal biological claims would require validation in an independent cohort and, where applicable, experimental evidence.

In [ ]:
local_enrichment = positional_log2_enrichment(local_sequences, labels == 1, labels == 0)
local_positions = np.arange(-24, 25)
local_entries = [
    {'residue': residue, 'relative_position': int(local_positions[column]), 'log2_enrichment': float(local_enrichment[row, column])}
    for row, residue in enumerate(AMINO_ACIDS)
    for column in range(local_enrichment.shape[1])
    if local_positions[column] != 0
]
most_enriched = sorted(local_entries, key=lambda item: item['log2_enrichment'], reverse=True)[:10]
most_depleted = sorted(local_entries, key=lambda item: item['log2_enrichment'])[:10]

radial_enrichment = radial_band_log2_enrichment(
    context_sequences, labels == 1, labels == 0, CONTEXT_BANDS
)
band_labels = [f'±{start}–{stop}' for start, stop in CONTEXT_BANDS]
radial_entries = [
    {'residue': residue, 'distance_band': band_labels[column], 'log2_enrichment': float(radial_enrichment[row, column])}
    for row, residue in enumerate(AMINO_ACIDS)
    for column in range(radial_enrichment.shape[1])
]
display({
    'largest_local_enrichments': most_enriched,
    'largest_local_depletions': most_depleted,
    'largest_broad_context_enrichments': sorted(radial_entries, key=lambda item: item['log2_enrichment'], reverse=True)[:10],
    'expert_correctness_counts': complementarity_counts,
})

biological_stems = {
    'figure_06_local_residue_enrichment',
    'figure_07_expert_complementarity_biology',
    'figure_08_context_radial_enrichment',
}
biological_files = {f'{stem}.{suffix}' for stem in biological_stems for suffix in ('pdf', 'svg', 'png')}
missing_biological_files = sorted(
    biological_files - {path.name for path in FIGURE_DIR.iterdir() if path.is_file()}
)
if missing_biological_files:
    raise RuntimeError(f'Missing biological figure exports: {missing_biological_files}')
biological_manifest = {
    'status': 'three biological-interpretation figures generated',
    'analysis_role': 'post hoc descriptive analysis; no causal or confirmatory claim',
    'cohort_records': len(labels),
    'matched_test_indices_sha256': representation_manifest['cohort']['matched_indices_sha256'],
    'classification_threshold': 0.5,
    'local_enrichment': {'window_size': 49, 'pseudocount': 0.5},
    'context_enrichment': {'window_size': 257, 'bands': [list(band) for band in CONTEXT_BANDS], 'pseudocount': 0.5},
    'expert_correctness_counts': complementarity_counts,
    'figures': {name: sha256_file(FIGURE_DIR / name) for name in sorted(biological_files)},
}
BIOLOGICAL_MANIFEST_PATH = VISUALIZATION_DIR / 'biological_interpretation_manifest.json'
BIOLOGICAL_MANIFEST_PATH.write_text(json.dumps(biological_manifest, indent=2, sort_keys=True) + '\n')
display({
    'biological_figure_files': sorted(biological_files),
    'manifest': str(BIOLOGICAL_MANIFEST_PATH),
    'interpretation_boundary': biological_manifest['analysis_role'],
})